# Voice Cloning Detection - Model Training
## AI-Powered Real-Time Detection of Voice Cloning Impersonation Attacks

This notebook trains the voice authenticity classifier using:
- **ASVspoof 2021** (bonafide vs spoofed audio)
- **WaveFake** (real vs AI-generated speech)

### Dataset Licenses
| Dataset | License | Registration Needed? |
|---------|---------|---------------------|
| **ASVspoof 2021** | Open Data Commons Attribution | **No** - free on Zenodo |
| **WaveFake** | CC BY 4.0 | **No** - free on HuggingFace |

> **Note:** ASVspoof 2019 and earlier require a signed license agreement. ASVspoof 2021 is openly licensed.

**Runtime:** Use `Runtime > Change runtime type > T4 GPU`

## 1. Install Dependencies

In [ ]:
!pip install -q librosa soundfile onnxruntime onnx tqdm scikit-learn matplotlib datasets

## 2. Clone Project & Setup

In [ ]:
import os
import sys
import torch
import numpy as np
from pathlib import Path

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Project structure
PROJECT_DIR = Path("voice_detection_app")
DATA_DIR = Path("data")
MODEL_DIR = PROJECT_DIR

for d in [PROJECT_DIR / "models", PROJECT_DIR / "services", PROJECT_DIR / "train", PROJECT_DIR / "edge", DATA_DIR / "enrollments"]:
    d.mkdir(parents=True, exist_ok=True)

# Create __init__.py files
for d in [PROJECT_DIR, PROJECT_DIR / "models", PROJECT_DIR / "services", PROJECT_DIR / "train", PROJECT_DIR / "edge"]:
    (d / "__init__.py").touch()

print("Project structure created.")

## 3. Define All Modules
We copy the source files directly into Colab to avoid import issues.

In [ ]:
%%writefile voice_detection_app/config.py
import os
from dataclasses import dataclass, field

@dataclass
class AudioConfig:
    sample_rate: int = 16000
    n_mfcc: int = 40
    n_fft: int = 2048
    hop_length: int = 512
    max_duration_sec: float = 30.0
    segment_duration_sec: float = 3.0

@dataclass
class ModelConfig:
    input_features: int = 64
    hidden_sizes: list = field(default_factory=lambda: [256, 128, 64])
    num_classes: int = 2
    dropout: float = 0.3
    model_path: str = "voice_detection_app/trained_model.pth"

@dataclass
class RiskConfig:
    high_risk_threshold: float = 0.75
    medium_risk_threshold: float = 0.45
    low_risk_threshold: float = 0.2
    context_multipliers: dict = field(default_factory=lambda: {
        "high_value_transaction": 1.3,
        "privileged_access": 1.25,
        "regular_call": 1.0,
    })

@dataclass
class AlertConfig:
    enabled: bool = True
    channels: list = field(default_factory=lambda: ["ui", "log"])
    webhook_url: str = ""
    email_recipients: list = field(default_factory=list)

@dataclass
class SpeakerConfig:
    enrollment_dir: str = "data/enrollments"
    verification_threshold: float = 0.55
    max_enrollment_samples: int = 10
    cosine_weight: float = 0.35
    zscore_weight: float = 0.25
    euclidean_weight: float = 0.20
    mahalanobis_weight: float = 0.20

@dataclass
class EdgeConfig:
    onnx_path: str = "voice_detection_app/trained_model.onnx"
    torchscript_path: str = "voice_detection_app/trained_model.pt"
    quantized_path: str = "voice_detection_app/trained_model_quantized.pt"
    opset_version: int = 14
    enable_quantization: bool = True

@dataclass
class StreamingConfig:
    segment_duration_sec: float = 3.0
    max_concurrent_sessions: int = 100
    session_timeout_sec: float = 600.0

@dataclass
class AppConfig:
    audio: AudioConfig = field(default_factory=AudioConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    risk: RiskConfig = field(default_factory=RiskConfig)
    alert: AlertConfig = field(default_factory=AlertConfig)
    speaker: SpeakerConfig = field(default_factory=SpeakerConfig)
    edge: EdgeConfig = field(default_factory=EdgeConfig)
    streaming: StreamingConfig = field(default_factory=StreamingConfig)
    host: str = "0.0.0.0"
    port: int = 8000
    debug: bool = True

settings = AppConfig()

In [ ]:
%%writefile voice_detection_app/models/detector.py
import numpy as np
import torch
import torch.nn as nn
from voice_detection_app.config import settings

class VoiceAuthenticityNet(nn.Module):
    def __init__(self, input_size: int = 64, hidden_sizes: list[int] | None = None, dropout: float = 0.3):
        super().__init__()
        if hidden_sizes is None:
            hidden_sizes = [256, 128, 64]
        layers = []
        prev_size = input_size
        for h_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, h_size),
                nn.BatchNorm1d(h_size),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_size = h_size
        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

class VoiceDetector:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = VoiceAuthenticityNet(
            input_size=settings.model.input_features,
            hidden_sizes=settings.model.hidden_sizes,
            dropout=settings.model.dropout,
        ).to(self.device)
        self.model.eval()
        self._trained = False

    @property
    def is_trained(self) -> bool:
        return self._trained

    def predict(self, feature_vector: np.ndarray) -> dict[str, float]:
        self.model.eval()
        with torch.no_grad():
            x = torch.tensor(feature_vector, dtype=torch.float32).unsqueeze(0).to(self.device)
            logit = self.model(x)
            prob = torch.sigmoid(logit).item()
        return {"synthetic_probability": prob, "genuine_probability": 1.0 - prob, "is_synthetic": prob > 0.5}

    def load_model(self, path: str | None = None):
        path = path or settings.model.model_path
        try:
            state = torch.load(path, map_location=self.device, weights_only=True)
            self.model.load_state_dict(state)
            self.model.eval()
            self._trained = True
        except FileNotFoundError:
            self._trained = False

    def save_model(self, path: str | None = None):
        path = path or settings.model.model_path
        torch.save(self.model.state_dict(), path)

In [ ]:
%%writefile voice_detection_app/services/audio_processor.py
import tempfile
from pathlib import Path
import librosa
import numpy as np
from voice_detection_app.config import settings

class AudioProcessor:
    def __init__(self):
        self.sr = settings.audio.sample_rate
        self.n_mfcc = settings.audio.n_mfcc
        self.n_fft = settings.audio.n_fft
        self.hop_length = settings.audio.hop_length
        self.max_duration = settings.audio.max_duration_sec
        self.segment_duration = settings.audio.segment_duration_sec

    def load_audio(self, file_path: str | Path) -> tuple[np.ndarray, int]:
        y, sr = librosa.load(str(file_path), sr=self.sr, duration=self.max_duration)
        return y, sr

    def load_audio_from_bytes(self, audio_bytes: bytes) -> tuple[np.ndarray, int]:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmp:
            tmp.write(audio_bytes)
            tmp.flush()
            return self.load_audio(tmp.name)

    def extract_mfcc(self, y: np.ndarray) -> np.ndarray:
        return librosa.feature.mfcc(y=y, sr=self.sr, n_mfcc=self.n_mfcc, n_fft=self.n_fft, hop_length=self.hop_length)

    def extract_spectral_features(self, y: np.ndarray) -> dict[str, float]:
        sc = librosa.feature.spectral_centroid(y=y, sr=self.sr)
        sb = librosa.feature.spectral_bandwidth(y=y, sr=self.sr)
        sr_ = librosa.feature.spectral_rolloff(y=y, sr=self.sr)
        scon = librosa.feature.spectral_contrast(y=y, sr=self.sr)
        zcr = librosa.feature.zero_crossing_rate(y)
        rms = librosa.feature.rms(y=y)
        return {
            "spectral_centroid_mean": float(np.mean(sc)),
            "spectral_centroid_std": float(np.std(sc)),
            "spectral_bandwidth_mean": float(np.mean(sb)),
            "spectral_bandwidth_std": float(np.std(sb)),
            "spectral_rolloff_mean": float(np.mean(sr_)),
            "spectral_rolloff_std": float(np.std(sr_)),
            "spectral_contrast_mean": float(np.mean(scon)),
            "zero_crossing_rate_mean": float(np.mean(zcr)),
            "rms_energy_mean": float(np.mean(rms)),
            "rms_energy_std": float(np.std(rms)),
        }

    def extract_prosody_features(self, y: np.ndarray) -> dict[str, float]:
        pitches, magnitudes = librosa.piptrack(y=y, sr=self.sr)
        pitch_values = pitches[magnitudes > np.median(magnitudes)]
        if len(pitch_values) == 0:
            pitch_values = np.array([0.0])
        onset_env = librosa.onset.onset_strength(y=y, sr=self.sr)
        tempo, _ = librosa.beat.beat_track(onset_envelope=onset_env, sr=self.sr)
        return {
            "pitch_mean": float(np.mean(pitch_values)),
            "pitch_std": float(np.std(pitch_values)),
            "pitch_range": float(np.ptp(pitch_values)),
            "tempo": float(tempo) if np.isscalar(tempo) else float(tempo[0]),
            "onset_strength_mean": float(np.mean(onset_env)),
            "onset_strength_std": float(np.std(onset_env)),
        }

    def extract_phase_features(self, y: np.ndarray) -> dict[str, float]:
        stft_mag = np.abs(librosa.stft(y, n_fft=self.n_fft, hop_length=self.hop_length))
        phase = np.angle(librosa.stft(y, n_fft=self.n_fft, hop_length=self.hop_length))
        phase_diff = np.diff(phase, axis=1)
        return {
            "phase_diff_mean": float(np.mean(np.abs(phase_diff))),
            "phase_diff_std": float(np.std(phase_diff)),
            "stft_energy_mean": float(np.mean(stft_mag)),
            "stft_energy_std": float(np.std(stft_mag)),
        }

    def extract_all_features(self, y: np.ndarray) -> dict[str, float]:
        features = {}
        features.update(self.extract_spectral_features(y))
        features.update(self.extract_prosody_features(y))
        features.update(self.extract_phase_features(y))
        return features

    def process_audio(self, y: np.ndarray) -> tuple[np.ndarray, dict[str, float]]:
        aggregated = self.extract_all_features(y)
        mfcc = self.extract_mfcc(y)
        aggregated["mfcc_mean"] = float(np.mean(mfcc))
        aggregated["mfcc_std"] = float(np.std(mfcc))
        return mfcc, aggregated

    def get_feature_vector(self, aggregated: dict[str, float], target_length: int = 64) -> np.ndarray:
        keys = sorted(aggregated.keys())
        values = [aggregated[k] for k in keys]
        if len(values) < target_length:
            values.extend([0.0] * (target_length - len(values)))
        elif len(values) > target_length:
            values = values[:target_length]
        return np.array(values, dtype=np.float32)

## 4. Download Datasets

### About ASVspoof 2021
- **What it is:** The world's largest benchmark dataset for detecting spoofed/fake speech
- **Content:** Real human speech (bonafide) + fake speech generated by TTS/VC/voice cloning
- **Format:** 16kHz FLAC audio files
- **License:** Open Data Commons Attribution (no registration needed)
- **Source:** Zenodo (https://zenodo.org/record/4837280)

### About WaveFake
- **What it is:** Dataset of AI-generated speech from multiple TTS systems
- **Content:** Fake audio from WaveNet, WaveRNN, MelGAN, HiFi-GAN, etc.
- **Format:** WAV audio files
- **License:** CC BY 4.0 (HuggingFace)

In [ ]:
#@title Download WaveFake Dataset (from Zenodo) { display-mode: "form" }
import subprocess
import zipfile
import os
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# WaveFake is on Zenodo: https://zenodo.org/records/5270336
# NOTE: Full dataset is ~23 GB. We'll also support HuggingFace streaming.
WAVEFAKE_ZENODO_URL = "https://zenodo.org/records/5270336/files/generated_audio.zip?download=1"

wavefake_dir = DATA_DIR / "wavefake"
wavefake_zip = DATA_DIR / "generated_audio.zip"

# Option 1: Try HuggingFace datasets library (smaller, easier)
use_hf = False
if not wavefake_dir.exists() or not any(wavefake_dir.rglob("*.wav")):
    try:
        from datasets import load_dataset
        print("Trying HuggingFace datasets library...")
        ds = load_dataset("ajaykarthick/wavefake-audio", split="train", streaming=True)
        wavefake_dir.mkdir(parents=True, exist_ok=True)
        count = 0
        for i, sample in enumerate(ds):
            if count >= 3000:  # Limit for manageable size
                break
            audio = sample["audio"]
            # Save as wav using soundfile
            import soundfile as sf
            audio_data = audio["array"]
            sr = audio["sampling_rate"]
            out_path = wavefake_dir / f"wavefake_{i:05d}.wav"
            sf.write(str(out_path), audio_data, sr)
            count += 1
            if count % 500 == 0:
                print(f"  Downloaded {count} files...")
        print(f"Downloaded {count} files from HuggingFace")
        use_hf = True
    except Exception as e:
        print(f"HuggingFace failed: {e}")
        print("Falling back to Zenodo (23 GB download)...")

# Option 2: Download from Zenodo
if not use_hf and (not wavefake_dir.exists() or not any(wavefake_dir.rglob("*.wav"))):
    print("Downloading WaveFake from Zenodo (~23 GB)...")
    print("This may take 10-30 minutes depending on your connection.")
    !wget -q --show-progress "{WAVEFAKE_ZENODO_URL}" -O "{wavefake_zip}"
    if os.path.getsize(wavefake_zip) > 1000:  # Verify it downloaded
        print("Extracting...")
        with zipfile.ZipFile(wavefake_zip, 'r') as z:
            z.extractall(DATA_DIR)
        os.remove(wavefake_zip)
        print("Done.")
    else:
        print("Download failed. File too small.")
        os.remove(wavefake_zip)
else:
    print("WaveFake already downloaded.")

# Count files
wav_count = len(list(wavefake_dir.rglob("*.wav")))
print(f"WaveFake: {wav_count} wav files found")

In [ ]:
#@title Download ASVspoof 2021 LA (Open Data - No Registration Required) { display-mode: "form" }
# =============================================================================
# ASVspoof 2021 License: Open Data Commons Attribution
# Source: https://zenodo.org/record/4837280
# No registration or license signing needed - freely downloadable.
# =============================================================================
import subprocess
import zipfile
import tarfile
from pathlib import Path
import os

DATA_DIR = Path("data")
asvspoof_dir = DATA_DIR / "asvspoof"
asvspoof_dir.mkdir(exist_ok=True)

# ASVspoof 2021 Logical Access (LA) dataset - hosted on Zenodo
# These URLs are direct download links - no auth needed
ASVSPOOF_URLS = {
    "train_flac": "https://zenodo.org/record/4837280/files/ASVspoof2021Train.zip",
    "dev_flac": "https://zenodo.org/record/4837280/files/ASVspoof2021Dev.zip",
    "eval_flac": "https://zenodo.org/record/4837280/files/ASVspoof2021Eval.zip",
}

# Labels/keys - hosted on asvspoof.org (publicly available)
LABEL_URLS = {
    "train_labels": "https://www.asvspoof.org/asvspoof2021/LA-keys-full.tar.gz",
}

print("ASVspoof 2021 - Open Data Commons Attribution License")
print("Source: https://zenodo.org/record/4837280")
print("=" * 60)

# Download audio files from Zenodo
for name, url in ASVSPOOF_URLS.items():
    fname = url.split("/")[-1]
    dest = asvspoof_dir / fname
    extracted_dir = asvspoof_dir / fname.replace(".zip", "")
    if extracted_dir.exists() and any(extracted_dir.rglob("*.flac")):
        print(f"[SKIP] {name} already extracted.")
        continue
    if not dest.exists():
        print(f"[DOWNLOAD] {name} (~1-2 GB each)...")
        try:
            !wget -q --show-progress "{url}" -O "{dest}"
        except Exception as e:
            print(f"  wget failed, trying curl...")
            !curl -L -o "{dest}" "{url}"
    print(f"[EXTRACT] {name}...")
    try:
        with zipfile.ZipFile(dest, 'r') as z:
            z.extractall(asvspoof_dir)
        os.remove(dest)
        print(f"  Done.")
    except Exception as e:
        print(f"  Extraction failed: {e}")

# Download labels
for name, url in LABEL_URLS.items():
    fname = url.split("/")[-1]
    dest = asvspoof_dir / fname
    extracted_dir = asvspoof_dir / fname.replace(".tar.gz", "")
    if extracted_dir.exists():
        print(f"[SKIP] {name} already extracted.")
        continue
    if not dest.exists():
        print(f"[DOWNLOAD] {name}...")
        !wget -q --show-progress "{url}" -O "{dest}"
    print(f"[EXTRACT] {name}...")
    try:
        with tarfile.open(dest, 'r:gz') as t:
            t.extractall(asvspoof_dir)
        os.remove(dest)
    except Exception as e:
        print(f"  Extraction failed: {e}")

print("\n" + "=" * 60)
print("ASVspoof 2021 download complete.")
flac_count = len(list(asvspoof_dir.rglob("*.flac")))
print(f"ASVspoof: {flac_count} flac files found")
print("\nDataset structure:")
for d in sorted(asvspoof_dir.iterdir()):
    if d.is_dir():
        count = len(list(d.rglob("*.flac")))
        print(f"  {d.name}/: {count} files")

## 5. Feature Extraction & Dataset Building

In [ ]:
#@title Extract features from all datasets { display-mode: "form" }
import os
import json
import numpy as np
import librosa
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

from voice_detection_app.services.audio_processor import AudioProcessor

processor = AudioProcessor()
DATA_DIR = Path("data")
FEATURE_CACHE = DATA_DIR / "feature_cache"
FEATURE_CACHE.mkdir(exist_ok=True)

FEATURE_FILE = FEATURE_CACHE / "features.npz"

if FEATURE_FILE.exists():
    print("Loading cached features...")
    data = np.load(FEATURE_FILE, allow_pickle=True)
    X_all = data["X"]
    y_all = data["y"]
    print(f"Loaded {len(X_all)} samples ({int(np.sum(y_all==0))} genuine, {int(np.sum(y_all==1))} synthetic)")
else:
    print("Extracting features (this may take 20-60 minutes)...")
    X_list = []
    y_list = []

    # --- WaveFake (synthetic = 1) ---
    wavefake_dir = DATA_DIR / "wavefake"
    if wavefake_dir.exists():
        wav_files = list(wavefake_dir.rglob("*.wav"))
        print(f"Processing {len(wav_files)} WaveFake files...")
        for f in tqdm(wav_files[:5000], desc="WaveFake"):
            try:
                y, sr = processor.load_audio(f)
                if len(y) < sr:
                    continue
                _, agg = processor.process_audio(y)
                fv = processor.get_feature_vector(agg, target_length=64)
                X_list.append(fv)
                y_list.append(1)
            except Exception:
                pass

    # --- ASVspoof (bonafide=0, spoof=1) ---
    asvspoof_dir = DATA_DIR / "asvspoof"
    if asvspoof_dir.exists():
        flac_files = list(asvspoof_dir.rglob("*.flac"))
        print(f"Processing {len(flac_files)} ASVspoof files...")
        for f in tqdm(flac_files[:8000], desc="ASVspoof"):
            try:
                y, sr = processor.load_audio(f)
                if len(y) < sr:
                    continue
                _, agg = processor.process_audio(y)
                fv = processor.get_feature_vector(agg, target_length=64)
                X_list.append(fv)
                # Determine label from path
                path_str = str(f).lower()
                if "bonafide" in path_str or "la_D_" in str(f):
                    y_list.append(0)
                else:
                    y_list.append(1)
            except Exception:
                pass

    X_all = np.array(X_list, dtype=np.float32)
    y_all = np.array(y_list, dtype=np.float32)

    # Save cache
    np.savez(FEATURE_FILE, X=X_all, y=y_all)
    print(f"Features cached to {FEATURE_FILE}")

print(f"\nDataset: {len(X_all)} samples")
print(f"  Genuine (0): {int(np.sum(y_all == 0))}")
print(f"  Synthetic (1): {int(np.sum(y_all == 1))}")
print(f"  Feature dimension: {X_all.shape[1]}")

## 6. Train the Model

In [ ]:
#@title Training Configuration { display-mode: "form" }
NUM_EPOCHS = 80 #@param {type:"integer"}
BATCH_SIZE = 64 #@param {type:"integer"}
LEARNING_RATE = 0.001 #@param {type:"number"}
DROPOUT = 0.3 #@param {type:"number"}
HIDDEN_SIZES = "256,128,64" #@param {type:"string"}

hidden = [int(x.strip()) for x in HIDDEN_SIZES.split(",")]
print(f"Config: epochs={NUM_EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}, dropout={DROPOUT}, hidden={hidden}")

In [ ]:
#@title Train { display-mode: "form" }
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from voice_detection_app.models.detector import VoiceAuthenticityNet

# Split data
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

# Data loaders
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# Model
model = VoiceAuthenticityNet(
    input_size=64,
    hidden_sizes=hidden,
    dropout=DROPOUT,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining on: {device}")

# Training loop
best_val_loss = float("inf")
best_accuracy = 0.0
patience = 0
MAX_PATIENCE = 15
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_X).squeeze(-1)
        loss = criterion(output, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        predicted = (torch.sigmoid(output) > 0.5).float()
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            output = model(batch_X).squeeze(-1)
            loss = criterion(output, batch_y)
            val_loss += loss.item()
            predicted = (torch.sigmoid(output) > 0.5).float()
            val_correct += (predicted == batch_y).sum().item()
            val_total += batch_y.size(0)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    train_acc = correct / total
    val_acc = val_correct / val_total
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f} | LR: {lr_now:.6f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_accuracy = val_acc
        patience = 0
        torch.save(model.state_dict(), "voice_detection_app/trained_model.pth")
    else:
        patience += 1
        if patience >= MAX_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete.")
print(f"Best val loss: {best_val_loss:.4f} | Best accuracy: {best_accuracy:.4f}")

## 7. Evaluate & Visualize

In [ ]:
#@title Plot Training Metrics { display-mode: "form" }
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(history["val_loss"], label="Val Loss", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history["val_acc"], label="Val Accuracy", color="green", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ROC Curve
model.eval()
all_probs = []
with torch.no_grad():
    for batch_X, _ in val_loader:
        batch_X = batch_X.to(device)
        output = model(batch_X).squeeze(-1)
        probs = torch.sigmoid(output).cpu().numpy()
        all_probs.extend(probs)

fpr, tpr, _ = roc_curve(y_val, all_probs)
roc_auc = auc(fpr, tpr)
axes[2].plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].set_title("ROC Curve")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

# Classification report
binary_preds = [1 if p > 0.5 else 0 for p in all_probs]
print("\nClassification Report:")
print(classification_report(y_val, binary_preds, target_names=["Genuine", "Synthetic"]))
print("Confusion Matrix:")
print(confusion_matrix(y_val, binary_preds))

## 8. Export to ONNX (Edge/Device Inference)

In [ ]:
#@title Export to ONNX { display-mode: "form" }
import torch
import onnx
from voice_detection_app.models.detector import VoiceAuthenticityNet

# Load best model
model = VoiceAuthenticityNet(input_size=64, hidden_sizes=hidden, dropout=0.0).to("cpu")
state = torch.load("voice_detection_app/trained_model.pth", map_location="cpu", weights_only=True)
model.load_state_dict(state)
model.eval()

# Export ONNX
dummy = torch.randn(1, 64)
onnx_path = "voice_detection_app/trained_model.onnx"

torch.onnx.export(
    model, dummy, onnx_path,
    opset_version=14,
    input_names=["features"],
    output_names=["synthetic_prob"],
    dynamic_axes={"features": {0: "batch"}, "synthetic_prob": {0: "batch"}},
)

# Validate
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

import os
size_mb = os.path.getsize(onnx_path) / 1e6
print(f"ONNX model exported: {onnx_path} ({size_mb:.2f} MB)")

# Quick ONNX inference test
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
test_input = np.random.randn(1, 64).astype(np.float32)
result = sess.run(None, {"features": test_input})
print(f"ONNX inference test: output={result[0].flatten()[0]:.4f}")
print("Export successful!")

## 9. Download Trained Models

In [ ]:
#@title Download models to your machine { display-mode: "form" }
from google.colab import files

files_to_download = [
    "voice_detection_app/trained_model.pth",
    "voice_detection_app/trained_model.onnx",
    "training_metrics.png",
]

for f in files_to_download:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)

print("\nDone! Copy these files to your local project:")
print("  trained_model.pth  -> voice_detection_app/trained_model.pth")
print("  trained_model.onnx -> voice_detection_app/trained_model.onnx")

## 10. Test Inference

In [ ]:
#@title Test with a sample audio file { display-mode: "form" }
import torch
import numpy as np
from voice_detection_app.models.detector import VoiceAuthenticityNet
from voice_detection_app.services.audio_processor import AudioProcessor

# Load model
model = VoiceAuthenticityNet(input_size=64, hidden_sizes=hidden, dropout=0.0).to("cpu")
state = torch.load("voice_detection_app/trained_model.pth", map_location="cpu", weights_only=True)
model.load_state_dict(state)
model.eval()

processor = AudioProcessor()

# Test with a random feature vector
test_fv = np.random.randn(64).astype(np.float32)
test_fv = (test_fv - test_fv.min()) / (test_fv.max() - test_fv.min())

with torch.no_grad():
    x = torch.tensor(test_fv).unsqueeze(0)
    prob = torch.sigmoid(model(x)).item()

print(f"Test prediction: synthetic_prob={prob:.4f}, label={'SYNTHETIC' if prob > 0.5 else 'GENUINE'}")
print("\nModel is working! Use it in your FastAPI app:")
print("  python -m voice_detection_app.app")